# Testing Notebook for the Classification Modules
This notebook serves as testing and implementation proof of the pixel based machine learning classification, utilizing full python environment.
The notebook is organize as follows:
1. Environment setup
2. Feature Selection
3. Hyperparameter Tuning and Model Evaluation
4. Classification and generate Land Cover Map

## Environment Setup

In [ ]:
#path setup
#since the notebook is in a subfolder, we need to add the src folder to the path
#The issue can be solved by installing the package in editable mode
from pathlib import Path
import sys
project_root = next(parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "src" / "ML_LC_Classifier").is_dir())
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [ ]:
#import the functions from the package
from ML_LC_Classifier import (load_and_split_training_data,
    select_features,
    tune_model,
    evaluate_model,
    classify_raster,
)

## Feature Elimination

In [ ]:
#define input path
landsat_input = r"C:\Agil_data\Project_EL\Landsat9_Final_Addtwi.tif"
sample_lc = r"C:\Agil_data\Project_EL\TrainingSamples_rev21.shp"
#conduct sample splitting
x_train, x_test, y_train, y_test = load_and_split_training_data(
    raster_path=landsat_input, 
    shapefile_path = sample_lc,
    class_field = 'LUCID',
    test_size = 0.4)

In [ ]:
#use XGBoost as the estimator for feature selection
#each classifier must have its own feature selection process, since the importance of features can vary between classifiers
from xgboost import XGBClassifier
xgb_estimator = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    random_state=42,
    n_jobs=-1,
)
#define the band names for easier interpretation of the results
band_name = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B10', 
             'elevation', 'slope', 'TPILF', 'aspect', 'FlowA', 
             'TWI', 'brightness', 'greenness', 'wetness', 'TCA', 
             'NDVI', 'BUI', 'MNDWI', 'EVI', 'AWEI']
#main execution for feature selection
selection_result = select_features(x_train, y_train, x_test,
                                    estimator=xgb_estimator,
                                    feature_names = band_name,
                                    cv=5, 
                                    scoring="balanced_accuracy",
                                    min_features_to_select=9,)
#reduce the training and testing data based on the selected features
x_train_selected = selection_result.X_train
x_test_selected = selection_result.X_test
selected_features = selection_result.selected_features
#check
print(selected_features)
print(x_train_selected.shape)

['B1', 'B2', 'B3', 'B4', 'B7', 'B10', 'elevation', 'TPILF', 'brightness', 'greenness', 'wetness', 'MNDWI', 'EVI', 'AWEI']
(8756, 14)


## Hyperparameter Tuning and Model Evaluation

In [ ]:
#Tune the selected-feature model and apply it to the full raster
import numpy as np
#define the parameter space
param_grid = {
    "n_estimators": [200, 400, 600],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.03, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}
#main execution for hyperparameter tuning
tuning_result = tune_model( X_train=x_train_selected, y_train=y_train, classifier = 'XGBoost',
    parameter_space=param_grid,
    search_method="random",
    n_iter=12,
    cv=5,
    scoring="balanced_accuracy",
    n_jobs=-1,
    random_state=42,
    verbose=0,
)
#check
print("Best parameters:", tuning_result.best_params)
print("Best CV score:", tuning_result.best_score)
#model evaluation based on the test data
test_eval = evaluate_model(tuning_result.model, x_test_selected, y_test,
    labels=np.unique(y_train),)
print("Test metrics:", test_eval.metrics)
print(test_eval.report["weighted avg"])


Best parameters: {'subsample': 0.8, 'n_estimators': 600, 'max_depth': 7, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
Best CV score: 0.9151683386921414
Test metrics: {'accuracy': 0.9638574854402192, 'balanced_accuracy': 0.9266454162141128, 'f1_macro': 0.9381456156758539, 'f1_weighted': 0.9636649195653066}
{'precision': 0.9638541882910271, 'recall': 0.9638574854402192, 'f1-score': 0.9636649195653066, 'support': 5838.0}


## Classification

In [7]:
output_dir = project_root / "output"
output_dir.mkdir(exist_ok=True)
output_raster = output_dir / "landcover_prediction.tif"

classify_raster(
    raster_path=landsat_input,
    model=tuning_result.model,
    output_path=output_raster,
    selector=selection_result.selector,
    block_size=512,
    nodata=0,
    output_dtype="int16",
)

print(f"Saved classified raster to: {output_raster}")

Saved classified raster to: c:\Users\AFahrezi\Documents\GitHub\Improve_pixel_based_lc_classification\output\landcover_prediction.tif
